# Ultimate SD Upscale — End-to-End Test

Tests every feature of the modular SDXL tiled upscale pipeline:
1. Basic linear upscale
2. Chess traversal
3. Gradient blending
4. Seam-fix band re-denoise
5. All combinations
6. Parameter validation
7. Parity check (single tile == full image)

## Setup

In [ ]:
# Install diffusers from your fork (modular-sdxl branch)
!pip install -q git+https://github.com/akshan-main/diffusers.git@modular-sdxl
!pip install -q transformers accelerate safetensors invisible_watermark

In [ ]:
import torch
import numpy as np
from PIL import Image
from diffusers import ModularPipeline
from diffusers.modular_pipelines.ultimate_sd_upscale.modular_blocks_ultimate_sd_upscale import (
    UltimateSDUpscaleBlocks,
)
from diffusers.utils import load_image
import time

print("Imports OK")

In [ ]:
# Load pipeline with SDXL weights
# Using SDXL-base for testing — swap for any SDXL checkpoint
blocks = UltimateSDUpscaleBlocks()
pipe = blocks.init_pipeline("stabilityai/stable-diffusion-xl-base-1.0")
pipe.load_components(torch_dtype=torch.float16)
pipe.to("cuda")

print("Pipeline loaded")
print(pipe.blocks)

In [ ]:
# Helper: create a test image (or load one)
def make_test_image(size=(512, 512)):
    """Generate a simple gradient test image for visual seam inspection."""
    w, h = size
    arr = np.zeros((h, w, 3), dtype=np.uint8)
    # Horizontal gradient (R)
    arr[:, :, 0] = np.linspace(0, 255, w, dtype=np.uint8)[np.newaxis, :]
    # Vertical gradient (G)
    arr[:, :, 1] = np.linspace(0, 255, h, dtype=np.uint8)[:, np.newaxis]
    # Diagonal gradient (B)
    arr[:, :, 2] = 128
    return Image.fromarray(arr)

def load_real_image():
    """Load a real photo for quality comparison."""
    url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/cat.png"
    img = load_image(url)
    # Resize to 512x512 as input
    return img.resize((512, 512), Image.LANCZOS)

# Display helper
def show_images(images, titles=None, figsize=(20, 8)):
    import matplotlib.pyplot as plt
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    for i, (img, ax) in enumerate(zip(images, axes)):
        ax.imshow(img)
        if titles:
            ax.set_title(titles[i], fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

# Prepare images
test_gradient = make_test_image((512, 512))
test_real = load_real_image()

show_images([test_gradient, test_real], ["Gradient test image", "Real test image"])

## Test 1: Basic Linear Upscale (2x)

Default settings: linear traversal, no blending, no seam fix.

In [ ]:
generator = torch.Generator(device="cuda").manual_seed(42)

t0 = time.time()
result_linear = pipe(
    prompt="high quality, detailed, sharp",
    image=test_real,
    upscale_factor=2.0,
    tile_size=512,
    tile_padding=32,
    strength=0.3,
    num_inference_steps=20,
    guidance_scale=7.5,
    generator=generator,
    output="images",
)
t1 = time.time()

img_linear = result_linear[0]
print(f"Output size: {img_linear.size}")
print(f"Time: {t1-t0:.1f}s")
assert img_linear.size == (1024, 1024), f"Expected 1024x1024, got {img_linear.size}"
show_images([test_real, img_linear], ["Input 512x512", "Linear 2x → 1024x1024"])

## Test 2: Chess Traversal

Same settings but with chess (checkerboard) traversal order.

In [ ]:
generator = torch.Generator(device="cuda").manual_seed(42)

t0 = time.time()
result_chess = pipe(
    prompt="high quality, detailed, sharp",
    image=test_real,
    upscale_factor=2.0,
    tile_size=512,
    tile_padding=32,
    traversal_mode="chess",
    strength=0.3,
    num_inference_steps=20,
    guidance_scale=7.5,
    generator=generator,
    output="images",
)
t1 = time.time()

img_chess = result_chess[0]
print(f"Output size: {img_chess.size}")
print(f"Time: {t1-t0:.1f}s")
show_images(
    [img_linear, img_chess],
    ["Linear traversal", "Chess traversal"],
)

## Test 3: Gradient Blending

Use gradient overlap blending to soften tile transitions.

In [ ]:
generator = torch.Generator(device="cuda").manual_seed(42)

t0 = time.time()
result_gradient = pipe(
    prompt="high quality, detailed, sharp",
    image=test_real,
    upscale_factor=2.0,
    tile_size=512,
    tile_padding=32,
    blend_mode="gradient",
    gradient_blend_overlap=16,
    strength=0.3,
    num_inference_steps=20,
    guidance_scale=7.5,
    generator=generator,
    output="images",
)
t1 = time.time()

img_gradient = result_gradient[0]
print(f"Output size: {img_gradient.size}")
print(f"Time: {t1-t0:.1f}s")

# Check no black borders (critical bugfix validation)
arr = np.array(img_gradient)
corner_vals = [arr[0,0].mean(), arr[0,-1].mean(), arr[-1,0].mean(), arr[-1,-1].mean()]
print(f"Corner pixel means: {corner_vals}")
assert all(v > 5 for v in corner_vals), f"Black border detected! Corners: {corner_vals}"
print("✓ No black borders")

show_images(
    [img_linear, img_gradient],
    ["No blending", "Gradient blending (overlap=16)"],
)

## Test 4: Seam-Fix Band Re-Denoise

Enable seam-fix: after the main tile pass, narrow bands along tile boundaries
are re-denoised with feathered mask blending.

In [ ]:
generator = torch.Generator(device="cuda").manual_seed(42)

t0 = time.time()
result_seamfix = pipe(
    prompt="high quality, detailed, sharp",
    image=test_real,
    upscale_factor=2.0,
    tile_size=512,
    tile_padding=32,
    seam_fix_width=64,
    seam_fix_padding=16,
    seam_fix_mask_blur=8,
    seam_fix_strength=0.3,
    strength=0.3,
    num_inference_steps=20,
    guidance_scale=7.5,
    generator=generator,
    output="images",
)
t1 = time.time()

img_seamfix = result_seamfix[0]
print(f"Output size: {img_seamfix.size}")
print(f"Time: {t1-t0:.1f}s (includes seam-fix pass)")
show_images(
    [img_linear, img_seamfix],
    ["No seam fix", "Seam fix (width=64, blur=8)"],
)

## Test 5: Chess + Gradient + Seam-Fix (All Features)

Full feature stack.

In [ ]:
generator = torch.Generator(device="cuda").manual_seed(42)

t0 = time.time()
result_all = pipe(
    prompt="high quality, detailed, sharp",
    image=test_real,
    upscale_factor=2.0,
    tile_size=512,
    tile_padding=32,
    traversal_mode="chess",
    blend_mode="gradient",
    gradient_blend_overlap=16,
    seam_fix_width=64,
    seam_fix_padding=16,
    seam_fix_mask_blur=8,
    seam_fix_strength=0.3,
    strength=0.3,
    num_inference_steps=20,
    guidance_scale=7.5,
    generator=generator,
    output="images",
)
t1 = time.time()

img_all = result_all[0]
print(f"Output size: {img_all.size}")
print(f"Time: {t1-t0:.1f}s")
show_images(
    [img_linear, img_chess, img_gradient, img_seamfix, img_all],
    ["Linear", "Chess", "Gradient blend", "Seam fix", "All features"],
    figsize=(25, 5),
)

## Test 6: Gradient Test Image (Seam Visibility Check)

The gradient image makes seams very visible. Compare with and without seam-fix.

In [ ]:
generator = torch.Generator(device="cuda").manual_seed(42)

result_grad_nosf = pipe(
    prompt="smooth gradient, abstract",
    image=test_gradient,
    upscale_factor=2.0,
    tile_size=512,
    tile_padding=32,
    strength=0.2,
    num_inference_steps=15,
    guidance_scale=5.0,
    generator=generator,
    output="images",
)

generator = torch.Generator(device="cuda").manual_seed(42)

result_grad_sf = pipe(
    prompt="smooth gradient, abstract",
    image=test_gradient,
    upscale_factor=2.0,
    tile_size=512,
    tile_padding=32,
    seam_fix_width=64,
    seam_fix_mask_blur=12,
    seam_fix_strength=0.2,
    strength=0.2,
    num_inference_steps=15,
    guidance_scale=5.0,
    generator=generator,
    output="images",
)

show_images(
    [test_gradient, result_grad_nosf[0], result_grad_sf[0]],
    ["Input gradient", "No seam fix (check seams)", "With seam fix"],
)

## Test 7: Parity Check — Single Tile Should Match Full Image

When tile_size is large enough to cover the entire image in one tile and seam-fix
is disabled, the output should be numerically close to standard SDXL img2img.

In [ ]:
# Small input so one tile covers everything
small_img = test_real.resize((256, 256), Image.LANCZOS)

# Upscale 2x → 512x512, tile_size=768 (> 512), so only 1 tile
generator = torch.Generator(device="cuda").manual_seed(123)

result_single_tile = pipe(
    prompt="high quality photo",
    image=small_img,
    upscale_factor=2.0,
    tile_size=768,
    tile_padding=0,
    strength=0.3,
    num_inference_steps=20,
    guidance_scale=7.5,
    generator=generator,
    output="images",
)

img_single = result_single_tile[0]
print(f"Single tile output: {img_single.size}")
assert img_single.size == (512, 512)

# Check output is not all black or all white
arr = np.array(img_single).astype(float)
print(f"Mean pixel value: {arr.mean():.1f}")
print(f"Std pixel value: {arr.std():.1f}")
assert arr.mean() > 10, "Output appears to be all black"
assert arr.mean() < 245, "Output appears to be all white"
assert arr.std() > 5, "Output appears to be flat/uniform"
print("✓ Parity check passed — output has reasonable pixel distribution")

show_images([small_img, img_single], ["Input 256x256", "Single-tile upscale 512x512"])

## Test 8: Determinism — Same Seed Same Output

In [ ]:
def run_deterministic(seed):
    gen = torch.Generator(device="cuda").manual_seed(seed)
    result = pipe(
        prompt="a cat sitting on a windowsill",
        image=test_real.resize((256, 256), Image.LANCZOS),
        upscale_factor=2.0,
        tile_size=768,
        tile_padding=0,
        strength=0.3,
        num_inference_steps=10,
        guidance_scale=7.5,
        generator=gen,
        output="images",
    )
    return np.array(result[0])

arr1 = run_deterministic(999)
arr2 = run_deterministic(999)

max_diff = np.abs(arr1.astype(float) - arr2.astype(float)).max()
print(f"Max pixel difference between identical runs: {max_diff}")
assert max_diff < 2, f"Non-deterministic! Max diff: {max_diff}"
print("✓ Determinism check passed")

## Test 9: Parameter Validation

In [ ]:
import traceback

def expect_error(name, fn):
    try:
        fn()
        print(f"✗ {name}: expected error but succeeded")
    except (ValueError, Exception) as e:
        print(f"✓ {name}: {type(e).__name__}: {e}")

# Invalid traversal mode
expect_error("Invalid traversal_mode='spiral'", lambda: pipe(
    prompt="test", image=test_real, traversal_mode="spiral",
    strength=0.3, num_inference_steps=5, output="images",
))

# Invalid blend_mode
expect_error("Invalid blend_mode='magic'", lambda: pipe(
    prompt="test", image=test_real, blend_mode="magic",
    strength=0.3, num_inference_steps=5, output="images",
))

# Negative tile_padding
expect_error("Negative tile_padding=-5", lambda: pipe(
    prompt="test", image=test_real, tile_padding=-5,
    strength=0.3, num_inference_steps=5, output="images",
))

# Negative seam_fix_padding
expect_error("Negative seam_fix_padding=-1", lambda: pipe(
    prompt="test", image=test_real, seam_fix_width=64, seam_fix_padding=-1,
    strength=0.3, num_inference_steps=5, output="images",
))

# Zero tile size
expect_error("Zero tile_size=0", lambda: pipe(
    prompt="test", image=test_real, tile_size=0,
    strength=0.3, num_inference_steps=5, output="images",
))

print("\n✓ All validation checks passed")

## Test 10: 4x Upscale (Stress Test)

256x256 → 1024x1024 with chess + seam-fix. Tests multiple tiles and seam bands.

In [ ]:
small = test_real.resize((256, 256), Image.LANCZOS)
generator = torch.Generator(device="cuda").manual_seed(42)

t0 = time.time()
result_4x = pipe(
    prompt="high quality, detailed, 4k photo",
    image=small,
    upscale_factor=4.0,
    tile_size=512,
    tile_padding=32,
    traversal_mode="chess",
    seam_fix_width=64,
    seam_fix_mask_blur=8,
    seam_fix_strength=0.25,
    strength=0.35,
    num_inference_steps=20,
    guidance_scale=7.5,
    generator=generator,
    output="images",
)
t1 = time.time()

img_4x = result_4x[0]
print(f"Output size: {img_4x.size}")
print(f"Time: {t1-t0:.1f}s")
assert img_4x.size == (1024, 1024)

show_images(
    [small, img_4x],
    ["Input 256x256", "4x upscale → 1024x1024 (chess + seam-fix)"],
)

## Test 11: Output Types

In [ ]:
small = test_real.resize((256, 256), Image.LANCZOS)

# PIL output (default)
gen = torch.Generator(device="cuda").manual_seed(42)
result_pil = pipe(
    prompt="test", image=small, upscale_factor=2.0, tile_size=768,
    tile_padding=0, strength=0.3, num_inference_steps=5,
    generator=gen, output="images", output_type="pil",
)
assert isinstance(result_pil[0], Image.Image)
print(f"✓ PIL output: {type(result_pil[0])}, size={result_pil[0].size}")

# NumPy output
gen = torch.Generator(device="cuda").manual_seed(42)
result_np = pipe(
    prompt="test", image=small, upscale_factor=2.0, tile_size=768,
    tile_padding=0, strength=0.3, num_inference_steps=5,
    generator=gen, output="images", output_type="np",
)
assert isinstance(result_np[0], np.ndarray)
print(f"✓ NumPy output: shape={result_np[0].shape}, dtype={result_np[0].dtype}")

# PyTorch output
gen = torch.Generator(device="cuda").manual_seed(42)
result_pt = pipe(
    prompt="test", image=small, upscale_factor=2.0, tile_size=768,
    tile_padding=0, strength=0.3, num_inference_steps=5,
    generator=gen, output="images", output_type="pt",
)
assert isinstance(result_pt[0], torch.Tensor)
print(f"✓ PyTorch output: shape={result_pt[0].shape}, dtype={result_pt[0].dtype}")

## Summary

In [ ]:
print("="*60)
print("END-TO-END TEST SUMMARY")
print("="*60)
print("Test 1:  Basic linear upscale        — check output above")
print("Test 2:  Chess traversal              — check output above")
print("Test 3:  Gradient blending            — check output above")
print("Test 4:  Seam-fix re-denoise          — check output above")
print("Test 5:  All features combined        — check output above")
print("Test 6:  Gradient seam visibility      — check output above")
print("Test 7:  Single-tile parity           — PASSED")
print("Test 8:  Determinism                  — PASSED")
print("Test 9:  Parameter validation         — PASSED")
print("Test 10: 4x upscale stress test       — check output above")
print("Test 11: Output types (pil/np/pt)     — PASSED")
print("="*60)
print("Visual tests require manual inspection of images above.")
print("Look for: tile boundary artifacts, black borders, seam lines.")